In [6]:
# ── 표준 라이브러리 ───────────────────────────────────────────────────
import os
import time
import torch   
import warnings
import re
import pathlib
warnings.filterwarnings("ignore")

# ── LangChain 핵심 컴포넌트 ──────────────────────────────────────────
from langchain_core.documents import Document
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# ── Retriever ────────────────────────────────────────────────────────
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# ── LangChain LCEL & Chain ────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

# ── 임베딩 모델 (로컬 Qwen3-Embedding) ───────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings

# ── LLM ──────────────────────────────────────────────────────────────
# [기본] 로컬 Ollama LLM 사용
from langchain_community.chat_models import ChatOllama

print("✅ 패키지 임포트 완료")

✅ 패키지 임포트 완료


In [ ]:
# %pip install -U "huggingface_hub[hf_xet]"
# !hf download Qwen/Qwen3-Embedding-0.6B

Note: you may need to restart the kernel to use updated packages.


In [5]:
# MPS 사용 가능하면 mps, 아니면 cpu
device = "mps" if torch.backends.mps.is_available() else "cpu"
print("사용 디바이스:", device)

embedding_model = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}  # 코사인 유사도 계산을 위해 정규화
)

print("✅ 임베딩 모델 로드 완료:", embedding_model.model_name)

사용 디바이스: mps


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

✅ 임베딩 모델 로드 완료: Qwen/Qwen3-Embedding-0.6B


TextSplitter은 문서의 구조를 모르기에 두가지 Risk Point 가 존재.

1. Section 2개가 1개의 Chunk에 섞인다.
 - ex : 10mg과 5mg이 같이 들어가서 LLM이 헷갈림

2. Section 제목과 내용이 떨어진다.
 - 청크만 봐서는 어느 목록인지 모름

위의 이유로 Size가 아닌 Section으로 자른다.

1. 섹션 제목 한 줄이 청크 안에 같이 들어 있으면 검색기는 그 단어로 맞는 청크를 더 위에 올리고 LLM은 그 단어를 읽고 어느 목록인지 판단하기 쉬움.

2. 검색할 때 섹션으로 거를 수 있다.
"퇴원 용량"을 물으면 Discharge Medications 청크만 찾게 할 수 있음

3. 근거 위치가 깔끔해진다.
"이 값은 원문 몇 번째 글자에서 나왔다"를 보여줘야 하므로 **"퇴원 약 목록 섹션의 이 줄"**이라고 정확히 가리킬 수 있음

In [10]:
# 섹션 제목 찾기: 줄 전체가 "제목:" 형태인 줄
# 예) "Discharge Medications:"  "Brief Hospital Course:"
HEADER_RE = re.compile(r"^([A-Z][A-Za-z /]+):\s*$", re.M)


def split_by_section(doc_text: str, doc_id: str,
                     size: int = 500, overlap: int = 50) -> list[Document]:
    """문서를 섹션 제목 위치에서 먼저 자르고, 긴 섹션만 글자 수로 한 번 더 자른다.
    반환되는 각 청크의 metadata:
      doc_id      어느 문서인지
      section     어느 섹션인지 (예: "Discharge Medications")
      char_start  원문 전체 기준 시작 위치
      char_end    원문 전체 기준 끝 위치
    """
    # 1) 섹션 제목 위치 찾기. 첫 제목 앞부분(이름·날짜 등)은 "Header"로 묶는다
    marks = [(0, "Header")] + [(m.start(), m.group(1))
                                 for m in HEADER_RE.finditer(doc_text)]

    inner = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=overlap, add_start_index=True
    )

    chunks = []
    for i, (start, name) in enumerate(marks):
        # 2) 이번 섹션의 끝 = 다음 섹션의 시작
        end = marks[i + 1][0] if i + 1 < len(marks) else len(doc_text)
        body = doc_text[start:end].rstrip()
        if not body.strip():
            continue

        # 3) 섹션이 size보다 길 때만 여러 조각으로 나뉜다
        for piece in inner.create_documents([body]):
            s = start + piece.metadata["start_index"]   # 섹션 기준 → 원문 기준
            e = s + len(piece.page_content)
            text = piece.page_content

            # 4) 긴 섹션이 나뉘어 제목이 떨어진 조각에는 제목을 다시 붙인다
            #    (char_start/char_end는 원문 위치를 그대로 가리킴)
            if name != "Header" and not text.startswith(name + ":"):
                text = f"{name}:\n{text}"

            chunks.append(Document(
                page_content=text,
                metadata={"doc_id": doc_id, "section": name,
                          "char_start": s, "char_end": e},
            ))
    return chunks

BM25: 키워드 매칭 기반 검색 (정확한 단어가 문서에 있을 때 강함)

VectorStore: 의미 기반 검색 (표현이 달라도 의미가 같으면 잘 찾음)

EnsembleRetriever: 두 검색기의 결과를 RRF(Reciprocal Rank Fusion)로 통합

In [13]:
chunks = []
for path in sorted(pathlib.Path("synthetic-kit/data/synthetic").glob("doc_*.txt")):
    chunks += split_by_section(path.read_text(encoding="utf-8"), path.stem, size=800)

vectorstore = Chroma.from_documents(chunks, embedding_model, collection_name="discharge")

dense = vectorstore.as_retriever(search_kwargs={"k": 5})
bm25 = BM25Retriever.from_documents(chunks, k=5)
hybrid = EnsembleRetriever(retrievers=[dense, bm25], weights=[0.5, 0.5])

In [14]:
q = "What is the discharge dose of lisinopril?"
for d in hybrid.invoke(q)[:3]:
    print(d.metadata["doc_id"], d.metadata["section"], d.metadata["char_start"])
    print(d.page_content[:120], "\n")

doc_03 Discharge Medications 2146
Discharge Medications:
1. Lisinopril 5 mg PO DAILY
2. Furosemide 40 mg PO DAILY
3. Metoprolol Tartrate 25 mg PO BID 

doc_09 Discharge Medications 1850
Discharge Medications:
1. Hydrochlorothiazide 25 mg q.d.
2. Carvedilol 12.5 mg twice daily
3. Rosuvastatin 10 mg QHS 

doc_10 Discharge Instructions 2473
Discharge Instructions:
You were admitted because of extra fluid and a fast heart rhythm. We
removed the fluid and your  

